# ChagaSight Final Ensemble Evaluation

**Objective:** Evaluate the complete 5-fold ensemble model on all validation data and compute comprehensive metrics for thesis documentation.

**Paper References:**
- Kim et al. (2025): Contour image embedding and REPA alignment
- Van Santvliet et al. (2025): Foundation model, demographics modulation, AoL, soft labels

**Methodology:**
1. Load all 5 trained fold models (fold0_best.pt through fold4_best.pt)
2. Run ensemble inference by averaging predictions across all models
3. Compute official PhysioNet Challenge metrics (TPR@5% with 10,000 permutations)
4. Compute threshold-based classification metrics (confusion matrix, accuracy, precision, recall, specificity, F1)
5. Generate visualizations (ROC curve, PR curve, calibration plot)
6. Save all results for thesis Chapter 8.3

**Features:**
- Checkpoint saving after each fold (can resume if interrupted)
- Optimized batch size (64) for RTX 3050 6GB
- All required metrics for thesis
- Professional visualizations

**Run this notebook AFTER all 5 folds are trained.**

In [1]:
# ==============================================================================
# Cell 1: Import Dependencies
# ==============================================================================

import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_curve, 
    precision_recall_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Configure matplotlib for professional plots
try:
    plt.style.use('seaborn-v0_8-paper')
except:
    plt.style.use('seaborn-paper')
sns.set_palette('husl')

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import official PhysioNet metrics
helper_code_path = project_root / 'external' / 'official_2025'
if str(helper_code_path) not in sys.path:
    sys.path.insert(0, str(helper_code_path))

OFFICIAL_METRICS = False
try:
    from helper_code import compute_challenge_score, compute_auc
    OFFICIAL_METRICS = True
    print("Status: Using OFFICIAL PhysioNet metrics (helper_code.py)")
except ImportError as e:
    print(f"Warning: helper_code.py not found: {e}")
    print("Expected location:", helper_code_path / 'helper_code.py')
    print("Will use sklearn metrics instead (results may differ slightly from official)")

# Import model and dataset
from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders

print("\nAll imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Status: Using OFFICIAL PhysioNet metrics (helper_code.py)

All imports successful.
PyTorch version: 2.7.1+cu118
NumPy version: 2.2.6
Pandas version: 2.3.3


In [2]:
# ==============================================================================
# Cell 2: Configuration and Setup
# ==============================================================================

# Directory paths
CHECKPOINT_DIR = project_root / 'checkpoints'
DATA_DIR = project_root / 'data' / 'processed'
METADATA_CSV = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR = DATA_DIR / '2d_images'
SIGNALS_DIR = DATA_DIR / '1d_signals_100hz'

# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: Running on CPU - this will be VERY slow!")

# Verify all 5 fold checkpoints exist
print("\nVerifying checkpoint files:")
fold_checkpoints = []
for fold in range(5):
    ckpt_path = CHECKPOINT_DIR / f'fold{fold}_best.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {ckpt_path}\n"
            f"Please ensure all 5 folds (fold0_best.pt through fold4_best.pt) are trained."
        )
    fold_checkpoints.append(ckpt_path)
    print(f"  Found: fold{fold}_best.pt ({ckpt_path.stat().st_size / 1024**2:.1f} MB)")

# Create evaluation checkpoint directory
EVAL_CHECKPOINT_DIR = CHECKPOINT_DIR / 'evaluation_checkpoints'
EVAL_CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f"\nEvaluation checkpoints will be saved in: {EVAL_CHECKPOINT_DIR}")

print("\nAll required checkpoints verified.")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory: 6.0 GB

Verifying checkpoint files:
  Found: fold0_best.pt (1987.7 MB)
  Found: fold1_best.pt (1987.7 MB)
  Found: fold2_best.pt (1987.7 MB)
  Found: fold3_best.pt (1987.7 MB)
  Found: fold4_best.pt (1987.7 MB)

Evaluation checkpoints will be saved in: d:\IIT\L6\FYP\ChagaSight\checkpoints\evaluation_checkpoints

All required checkpoints verified.


In [3]:
# ==============================================================================
# Cell 3: Load All 5 Fold Models
# ==============================================================================

print("Loading all 5 trained models...\n")

models = []
fold_scores = []

for fold in range(5):
    print(f"Loading Fold {fold} model...")
    
    # Initialize model architecture
    model = HybridChagasModel(
        img_size=(24, 2048),
        patch_size_2d=(8, 64),
        num_leads=12,
        seq_len_1d=1000,
        patch_size_1d=50,
        embed_dim=768,
        depth=12,
        num_heads=12,
        use_aol=True,
        use_demographics=True
    )
    
    # Load trained weights (weights_only=False for PyTorch 2.6+)
    checkpoint = torch.load(fold_checkpoints[fold], map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Move to device and set to eval mode
    model = model.to(device)
    model.eval()
    
    # Verify model is on correct device
    model_device = str(next(model.parameters()).device)
    assert device in model_device, f"Model {fold} not on {device}! Got {model_device}"
    
    models.append(model)
    
    # Record validation score from training
    val_score = checkpoint.get('val_score', 0.0)
    fold_scores.append(val_score)
    
    print(f"  Training validation TPR@5%: {val_score:.4f}")
    print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")
    print(f"  Phase: {checkpoint.get('phase', 'unknown')}")
    print(f"  Device: {model_device}\n")

# Model statistics
total_params = sum(p.numel() for p in models[0].parameters())
trainable_params = sum(p.numel() for p in models[0].parameters() if p.requires_grad)

print("="*70)
print(" MODEL STATISTICS")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024**2:.1f} MB (float32)")
print(f"\nIndividual fold scores:")
for i, score in enumerate(fold_scores):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print("\nAll 5 models loaded successfully.")
print("="*70)

Loading all 5 trained models...

Loading Fold 0 model...
  Training validation TPR@5%: 0.4027
  Epoch: unknown
  Phase: 2
  Device: cuda:0

Loading Fold 1 model...
  Training validation TPR@5%: 0.4165
  Epoch: unknown
  Phase: 2
  Device: cuda:0

Loading Fold 2 model...
  Training validation TPR@5%: 0.4332
  Epoch: unknown
  Phase: 2
  Device: cuda:0

Loading Fold 3 model...
  Training validation TPR@5%: 0.4288
  Epoch: unknown
  Phase: 2
  Device: cuda:0

Loading Fold 4 model...
  Training validation TPR@5%: 0.4291
  Epoch: unknown
  Phase: 2
  Device: cuda:0

 MODEL STATISTICS
Total parameters: 173,570,817
Trainable parameters: 173,570,817
Model size: 662.1 MB (float32)

Individual fold scores:
  Fold 0: 0.4027
  Fold 1: 0.4165
  Fold 2: 0.4332
  Fold 3: 0.4288
  Fold 4: 0.4291

Mean: 0.4221 ± 0.0112

All 5 models loaded successfully.


In [4]:
# ==============================================================================
# Cell 4: Run Ensemble Inference with Checkpoint Saving
# ==============================================================================
# CRITICAL: batch_size=64 for optimal performance on RTX 3050 6GB
# Saves results after each fold to allow resuming if interrupted
# ==============================================================================

print("Running ensemble inference with checkpoint saving...\n")
print("Each fold's validation set is evaluated with the ensemble of ALL 5 models.")
print("Results are saved after each fold completes.\n")

# Check GPU memory
if device == 'cuda':
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB\n")

# Check which folds are already completed
completed_folds = []
for fold in range(5):
    fold_checkpoint = EVAL_CHECKPOINT_DIR / f'fold{fold}_eval.npz'
    if fold_checkpoint.exists():
        completed_folds.append(fold)
        print(f"✓ Found existing results for Fold {fold}")

if completed_folds:
    print(f"\nResuming from fold {max(completed_folds) + 1}")
    print(f"To restart from scratch: delete {EVAL_CHECKPOINT_DIR}/*.npz\n")
else:
    print("Starting fresh evaluation (no existing checkpoints)\n")

# Accumulators
all_probs = []
all_labels = []
all_ids = []
all_datasets = []
all_folds = []

import time
total_start = time.time()

# Process each fold
for fold in range(5):
    fold_checkpoint = EVAL_CHECKPOINT_DIR / f'fold{fold}_eval.npz'
    
    # Skip if already completed
    if fold in completed_folds:
        print(f"{'='*70}")
        print(f" FOLD {fold}: Loading from checkpoint (already completed)")
        print(f"{'='*70}")
        
        data = np.load(fold_checkpoint, allow_pickle=True)
        all_probs.extend(data['probs'])
        all_labels.extend(data['labels'])
        all_ids.extend(data['ids'])
        all_datasets.extend(data['datasets'])
        all_folds.extend([fold] * len(data['labels']))
        
        n_pos = int(data['labels'].sum())
        n_total = len(data['labels'])
        print(f"Loaded {n_total:,} samples ({n_pos} positive, {n_total-n_pos} negative)\n")
        continue
    
    print(f"{'='*70}")
    print(f" FOLD {fold}: Running ensemble inference")
    print(f"{'='*70}")
    fold_start = time.time()
    
    # Create validation dataloader
    # CRITICAL: batch_size=64 for optimal speed (NOT 16 or 32!)
    _, val_loader = create_dataloaders(
        metadata_csv=str(METADATA_CSV),
        images_dir=str(IMAGES_DIR),
        signals_dir=str(SIGNALS_DIR),
        fold=fold,
        batch_size=64,     # Optimal for RTX 3050 6GB
        num_workers=0,     # Windows compatibility (CRITICAL)
        use_weighted_sampling=False,  # Evaluate on real distribution
        augment_train=False  # No augmentation for evaluation
    )
    
    fold_probs = []
    fold_labels = []
    fold_ids = []
    fold_datasets = []
    
    warmup_done = False
    
    # Inference loop
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(val_loader, desc=f"Fold {fold}", leave=True)):
            batch_start = time.time()
            
            # Move inputs to GPU (non_blocking for async transfer)
            images = batch['image'].to(device, non_blocking=True)
            signals = batch['signal'].to(device, non_blocking=True)
            ages = batch['age'].to(device, non_blocking=True)
            sexes = batch['sex'].to(device, non_blocking=True)
            
            # Get ground truth labels
            hard_labels = batch['hard_label'].cpu().numpy()
            ids = batch['id']
            datasets = batch['dataset']
            
            # Run all 5 models and collect predictions
            batch_preds = []
            for model in models:
                outputs = model(images, signals, ages, sexes)
                probs = torch.sigmoid(outputs['logits']).cpu().numpy()
                batch_preds.append(probs)
            
            # Ensemble: Average predictions across all 5 models
            ensemble_probs = np.mean(np.stack(batch_preds, axis=0), axis=0)
            
            # Accumulate results
            fold_probs.extend(ensemble_probs)
            fold_labels.extend(hard_labels)
            fold_ids.extend(ids)
            fold_datasets.extend(datasets)
            
            # Progress tracking for first 3 batches
            if batch_idx < 3 or not warmup_done:
                batch_time = time.time() - batch_start
                print(f"  Batch {batch_idx+1}: {batch_time:.2f}s ({len(hard_labels)} samples)")
                if batch_idx == 2:
                    warmup_done = True
                    avg_time = batch_time
                    est_time = avg_time * len(val_loader) / 60
                    print(f"  Estimated time for this fold: {est_time:.1f} minutes")
    
    # Convert to arrays
    fold_probs = np.array(fold_probs)
    fold_labels = np.array(fold_labels)
    
    # SAVE CHECKPOINT (critical for resuming if interrupted)
    np.savez(
        fold_checkpoint,
        probs=fold_probs,
        labels=fold_labels,
        ids=fold_ids,
        datasets=fold_datasets
    )
    print(f"\n✓ Checkpoint saved: {fold_checkpoint.name}")
    
    # Add to global accumulators
    all_probs.extend(fold_probs)
    all_labels.extend(fold_labels)
    all_ids.extend(fold_ids)
    all_datasets.extend(fold_datasets)
    all_folds.extend([fold] * len(fold_labels))
    
    # Fold summary
    n_pos = int(fold_labels.sum())
    n_total = len(fold_labels)
    fold_time = time.time() - fold_start
    
    print(f"\nFold {fold} Complete:")
    print(f"  Samples: {n_total:,} ({n_pos} positive, {n_total-n_pos} negative)")
    print(f"  Time: {fold_time/60:.1f} minutes")
    print(f"  Speed: {fold_time/n_total:.3f} seconds/sample\n")
    
    # Clear GPU cache between folds
    if device == 'cuda':
        torch.cuda.empty_cache()

# Convert to numpy arrays
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

total_time = time.time() - total_start

# Final summary
print("\n" + "="*70)
print(" ENSEMBLE INFERENCE COMPLETE")
print("="*70)
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"Total samples: {len(all_labels):,}")
print(f"Positive samples: {int(all_labels.sum()):,} ({100*all_labels.mean():.2f}%)")
print(f"Negative samples: {len(all_labels) - int(all_labels.sum()):,} ({100*(1-all_labels.mean()):.2f}%)")
print(f"Average time/sample: {total_time/len(all_labels):.3f} seconds")

# Per-dataset breakdown
print("\nPer-dataset composition:")
for dataset_name in ['ptbxl', 'samitrop', 'code15']:
    mask = np.array([d == dataset_name for d in all_datasets])
    if mask.sum() > 0:
        n_total_ds = mask.sum()
        n_pos_ds = all_labels[mask].sum()
        print(f"  {dataset_name.upper()}: {n_total_ds:,} samples ({n_pos_ds:.0f} positive, {100*n_pos_ds/n_total_ds:.2f}%)")

print(f"\nCheckpoints saved in: {EVAL_CHECKPOINT_DIR}")
print("If interrupted, re-run this cell to resume from last completed fold.")
print("="*70)

Running ensemble inference with checkpoint saving...

Each fold's validation set is evaluated with the ensemble of ALL 5 models.
Results are saved after each fold completes.

GPU memory allocated: 5.18 GB
GPU memory reserved: 5.73 GB

Starting fresh evaluation (no existing checkpoints)

 FOLD 0: Running ensemble inference


d:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


✓ Loaded train fold 0: 66504 samples
  Datasets: {'ptbxl': 17439, 'samitrop': 1305, 'code15': 47760}
  Positive: 2277, Negative: 64227
✓ Loaded val fold 0: 16626 samples
  Datasets: {'ptbxl': 4360, 'samitrop': 326, 'code15': 11940}
  Positive: 569, Negative: 16057

✓ Created dataloaders for fold 0:
  Train: 66504 samples, 1039 batches
  Val:   16626 samples, 260 batches
  Weighted sampling: False
  Augmentation: False


d:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


Fold 0:   0%|          | 0/260 [00:00<?, ?it/s]

  Batch 1: 61.53s (64 samples)
  Batch 2: 59.15s (64 samples)
  Batch 3: 63.21s (64 samples)
  Estimated time for this fold: 273.9 minutes


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# Cell 5: Compute Official PhysioNet Metrics
# ==============================================================================
# PRIMARY METRIC: TPR@5% (True Positive Rate at 5% capacity)
# ==============================================================================

print("\n" + "="*70)
print(" COMPUTING OFFICIAL METRICS")
print("="*70)

if OFFICIAL_METRICS:
    print("\nUsing OFFICIAL PhysioNet implementation...")
    print("Computing TPR@5% with 10,000 permutations (takes ~30 seconds)...\n")
    
    # Primary metric: TPR@5% with 10,000 permutations
    tpr_5pct = compute_challenge_score(
        labels=all_labels.astype(np.float64),
        outputs=all_probs.astype(np.float64),
        fraction_capacity=0.05,
        num_permutations=10000,
        seed=12345
    )
    
    # Secondary metrics
    auroc, auprc = compute_auc(all_labels, all_probs)
    
else:
    print("\nUsing sklearn metrics (approximate - results may differ from official)...\n")
    
    # Approximate TPR@5% using ROC curve
    fpr, tpr_roc, _ = roc_curve(all_labels, all_probs)
    idx = np.where(fpr <= 0.05)[0]
    tpr_5pct = float(tpr_roc[idx[-1]]) if len(idx) > 0 else 0.0
    
    # Secondary metrics
    auroc = roc_auc_score(all_labels, all_probs)
    auprc = average_precision_score(all_labels, all_probs)

# Display results
print("="*70)
print(" PRIMARY METRICS - All 5 Folds Combined")
print("="*70)
print(f"  TPR@5%:  {tpr_5pct:.4f}  ⭐ PRIMARY METRIC")
print(f"  AUROC:   {auroc:.4f}")
print(f"  AUPRC:   {auprc:.4f}")
print("="*70)

# Performance benchmarks
RANDOM_BASELINE = 0.050
TARGET_SCORE = 0.420
TOP_TEAM = 0.445
SOTA = 0.490

# Clinical metrics
n_total = len(all_labels)
n_pos = int(all_labels.sum())
capacity = int(0.05 * n_total)
cases_found = int(tpr_5pct * n_pos)
random_cases = int(RANDOM_BASELINE * n_pos)

print("\nClinical Interpretation:")
print(f"  Total patients: {n_total:,}")
print(f"  Chagas positive: {n_pos:,} ({100*n_pos/n_total:.2f}%)")
print(f"  Screening capacity (5%): {capacity:,} patients")
print(f"  Cases found by model: {cases_found:,} ({100*cases_found/n_pos:.1f}% of all Chagas)")
print(f"  Cases found by random: {random_cases:,} (baseline)")
print(f"  Improvement over random: {cases_found/random_cases:.1f}x")

# Performance assessment
print("\nPerformance Assessment:")
if tpr_5pct >= TOP_TEAM:
    print(f"  🎉 EXCELLENT: Matches/exceeds top competition team ({TOP_TEAM:.3f})!")
    print(f"  Margin: +{tpr_5pct - TOP_TEAM:.4f}")
elif tpr_5pct >= TARGET_SCORE:
    print(f"  ✓ GOOD: Target achieved ({TARGET_SCORE:.3f})!")
    print(f"  Margin above target: +{tpr_5pct - TARGET_SCORE:.4f}")
    print(f"  Gap to top team: -{TOP_TEAM - tpr_5pct:.4f}")
else:
    print(f"  ⚠️  Below target ({TARGET_SCORE:.3f})")
    print(f"  Gap: -{TARGET_SCORE - tpr_5pct:.4f}")
    print(f"  Consider: (1) Verify pretraining, (2) Check soft labels, (3) Increase dataset size")

print(f"\n  Performance vs SOTA: {100*tpr_5pct/SOTA:.1f}% of Van Santvliet et al. ({SOTA:.3f})")
print("="*70)

# Store for later cells
metrics_dict = {
    'tpr_5pct': float(tpr_5pct),
    'auroc': float(auroc),
    'auprc': float(auprc),
    'n_total': n_total,
    'n_positive': n_pos,
    'n_negative': n_total - n_pos,
    'using_official': OFFICIAL_METRICS
}

In [ ]:
# ==============================================================================
# Cell 6: Compute Threshold-Based Classification Metrics
# ==============================================================================
# For thesis Chapter 8.3: Confusion matrix, accuracy, precision, recall,
# specificity, F1 score at optimal threshold
# ==============================================================================

print("\n" + "="*70)
print(" THRESHOLD-BASED CLASSIFICATION METRICS")
print("="*70)

# Find optimal thresholds using different strategies
thresholds_to_test = {}

# Strategy 1: Default threshold
thresholds_to_test['default_0.5'] = 0.5

# Strategy 2: Optimal F1 score
precisions, recalls, pr_thresholds = precision_recall_curve(all_labels, all_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
optimal_f1_idx = np.argmax(f1_scores)
thresholds_to_test['optimal_f1'] = pr_thresholds[optimal_f1_idx]

# Strategy 3: Optimal Youden's J (Sensitivity + Specificity - 1)
fpr, tpr_roc, roc_thresholds = roc_curve(all_labels, all_probs)
j_scores = tpr_roc - fpr
optimal_j_idx = np.argmax(j_scores)
thresholds_to_test['optimal_j'] = roc_thresholds[optimal_j_idx]

print("\nThreshold strategies:")
print(f"  Default (0.5):           {thresholds_to_test['default_0.5']:.4f}")
print(f"  Optimal F1:              {thresholds_to_test['optimal_f1']:.4f}")
print(f"  Optimal Youden's J:      {thresholds_to_test['optimal_j']:.4f}")

# Compute metrics at each threshold
results_by_threshold = {}

for name, threshold in thresholds_to_test.items():
    # Binarize predictions
    y_pred = (all_probs >= threshold).astype(int)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(all_labels, y_pred).ravel()
    
    # Classification metrics
    accuracy = accuracy_score(all_labels, y_pred)
    precision = precision_score(all_labels, y_pred, zero_division=0)
    recall = recall_score(all_labels, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(all_labels, y_pred, zero_division=0)
    
    # PPV and NPV
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    
    results_by_threshold[name] = {
        'threshold': threshold,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'specificity': float(specificity),
        'f1_score': float(f1),
        'ppv': float(ppv),
        'npv': float(npv)
    }

# Use Optimal Youden's J as primary (balances sensitivity/specificity)
primary_threshold = 'optimal_j'
primary_results = results_by_threshold[primary_threshold]

# Display primary results
print("\n" + "="*70)
print(f" METRICS AT OPTIMAL THRESHOLD ({primary_results['threshold']:.4f})")
print("="*70)

print("\nConfusion Matrix:")
print("                      Predicted")
print("                 Negative    Positive")
print("            ┌─────────────────────────┐")
print(f"  Actual    │                         │")
print(f"  Negative  │  {primary_results['TN']:7,}    {primary_results['FP']:7,}  │")
print(f"  Positive  │  {primary_results['FN']:7,}    {primary_results['TP']:7,}  │")
print("            └─────────────────────────┘")

print("\nClassification Metrics:")
print(f"  Accuracy:    {primary_results['accuracy']:.4f}  ({100*primary_results['accuracy']:.2f}%)")
print(f"  Precision:   {primary_results['precision']:.4f}  (PPV - of predicted positive, {100*primary_results['precision']:.1f}% are correct)")
print(f"  Recall:      {primary_results['recall']:.4f}  (Sensitivity - of actual positive, {100*primary_results['recall']:.1f}% detected)")
print(f"  Specificity: {primary_results['specificity']:.4f}  (TNR - of actual negative, {100*primary_results['specificity']:.1f}% correct)")
print(f"  F1 Score:    {primary_results['f1_score']:.4f}")
print(f"  NPV:         {primary_results['npv']:.4f}")

print("\nClinical Outcomes:")
print(f"  True Positives (TP = {primary_results['TP']:,}): Correctly identified Chagas patients")
print(f"  False Negatives (FN = {primary_results['FN']:,}): Missed Chagas cases ({100*primary_results['FN']/n_pos:.1f}% missed)")
print(f"  False Positives (FP = {primary_results['FP']:,}): Healthy patients flagged ({100*primary_results['FP']/(n_total-n_pos):.2f}% FPR)")
print(f"  True Negatives (TN = {primary_results['TN']:,}): Correctly identified healthy patients")
print("="*70)

# Add to metrics dict
metrics_dict.update({
    'threshold': primary_results['threshold'],
    'confusion_matrix_tp': primary_results['TP'],
    'confusion_matrix_tn': primary_results['TN'],
    'confusion_matrix_fp': primary_results['FP'],
    'confusion_matrix_fn': primary_results['FN'],
    'accuracy': primary_results['accuracy'],
    'precision': primary_results['precision'],
    'recall': primary_results['recall'],
    'specificity': primary_results['specificity'],
    'f1_score': primary_results['f1_score']
})

In [ ]:
# ==============================================================================
# Cell 7: Per-Dataset Performance Analysis
# ==============================================================================

print("\n" + "="*70)
print(" PER-DATASET PERFORMANCE ANALYSIS")
print("="*70)

dataset_metrics = {}

for dataset_name in ['ptbxl', 'samitrop', 'code15']:
    mask = np.array([d == dataset_name for d in all_datasets])
    
    if mask.sum() == 0:
        continue
    
    ds_labels = all_labels[mask]
    ds_probs = all_probs[mask]
    
    # Skip if no positive samples
    if ds_labels.sum() == 0:
        print(f"\n{dataset_name.upper()}:")
        print(f"  Samples: {len(ds_labels):,} (all negative - used as controls)")
        continue
    
    # Compute metrics
    if OFFICIAL_METRICS:
        ds_tpr = compute_challenge_score(
            labels=ds_labels.astype(np.float64),
            outputs=ds_probs.astype(np.float64),
            fraction_capacity=0.05,
            num_permutations=10000,
            seed=12345
        )
        ds_auroc, ds_auprc = compute_auc(ds_labels, ds_probs)
    else:
        fpr_ds, tpr_ds, _ = roc_curve(ds_labels, ds_probs)
        idx_ds = np.where(fpr_ds <= 0.05)[0]
        ds_tpr = float(tpr_ds[idx_ds[-1]]) if len(idx_ds) > 0 else 0.0
        ds_auroc = roc_auc_score(ds_labels, ds_probs)
        ds_auprc = average_precision_score(ds_labels, ds_probs)
    
    n_pos_ds = int(ds_labels.sum())
    n_total_ds = len(ds_labels)
    
    print(f"\n{dataset_name.upper()}:")
    print(f"  Samples: {n_total_ds:,} ({n_pos_ds:,} positive, {100*n_pos_ds/n_total_ds:.2f}%)")
    print(f"  TPR@5%:  {ds_tpr:.4f}")
    print(f"  AUROC:   {ds_auroc:.4f}")
    print(f"  AUPRC:   {ds_auprc:.4f}")
    
    dataset_metrics[dataset_name] = {
        'n_samples': n_total_ds,
        'n_positive': n_pos_ds,
        'tpr_5pct': float(ds_tpr),
        'auroc': float(ds_auroc),
        'auprc': float(ds_auprc)
    }

print("\nNotes:")
print("  PTB-XL: Negative-only dataset (healthy controls)")
print("  SaMi-Trop: Verified diagnoses (gold standard labels)")
print("  CODE-15: ML-predicted labels (soft labels 0.2/0.8 in training)")
print("="*70)

In [ ]:
# ==============================================================================
# Cell 8: Generate Visualization Plots
# ==============================================================================

print("\nGenerating visualization plots...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('ChagaSight Ensemble Evaluation Results', fontsize=16, fontweight='bold')

# Plot 1: ROC Curve
ax = axes[0, 0]
fpr_plot, tpr_plot, _ = roc_curve(all_labels, all_probs)
ax.plot(fpr_plot, tpr_plot, linewidth=2, label=f'Ensemble (AUROC = {auroc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUROC = 0.500)')

# Highlight 5% FPR point
idx_5pct = np.argmin(np.abs(fpr_plot - 0.05))
ax.plot(fpr_plot[idx_5pct], tpr_plot[idx_5pct], 'ro', markersize=8,
        label=f'5% FPR (TPR = {tpr_plot[idx_5pct]:.3f})')

ax.set_xlabel('False Positive Rate (FPR)', fontsize=11)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=11)
ax.set_title('ROC Curve', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

# Plot 2: Precision-Recall Curve
ax = axes[0, 1]
precision_plot, recall_plot, _ = precision_recall_curve(all_labels, all_probs)
random_precision = all_labels.mean()

ax.plot(recall_plot, precision_plot, linewidth=2,
        label=f'Ensemble (AUPRC = {auprc:.3f})')
ax.axhline(y=random_precision, color='k', linestyle='--', linewidth=1,
          label=f'Random (AUPRC = {random_precision:.3f})')

ax.set_xlabel('Recall (Sensitivity)', fontsize=11)
ax.set_ylabel('Precision (PPV)', fontsize=11)
ax.set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

# Plot 3: Calibration Plot
ax = axes[1, 0]
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_true_rates = np.zeros(n_bins)

for i in range(n_bins):
    mask_bin = (all_probs >= bin_edges[i]) & (all_probs < bin_edges[i+1])
    if i == n_bins - 1:
        mask_bin = (all_probs >= bin_edges[i]) & (all_probs <= bin_edges[i+1])
    
    if mask_bin.sum() > 0:
        bin_true_rates[i] = all_labels[mask_bin].mean()

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.plot(bin_centers, bin_true_rates, 'o-', linewidth=2, markersize=6,
       label='Ensemble predictions')

ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Actual Positive Rate', fontsize=11)
ax.set_title('Calibration Plot', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

# Plot 4: Per-Fold Performance
ax = axes[1, 1]
fold_labels = [f'Fold {i}' for i in range(5)] + ['Ensemble']
fold_values = fold_scores + [tpr_5pct]
colors = ['#1f77b4'] * 5 + ['#ff7f0e']

bars = ax.bar(range(len(fold_labels)), fold_values, color=colors, alpha=0.8, edgecolor='black')
ax.axhline(y=TARGET_SCORE, color='g', linestyle='--', linewidth=1.5,
          label=f'Target ({TARGET_SCORE:.3f})')
ax.axhline(y=TOP_TEAM, color='r', linestyle='--', linewidth=1.5,
          label=f'Top Team ({TOP_TEAM:.3f})')

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('TPR@5%', fontsize=11)
ax.set_title('Per-Fold Performance', fontsize=12, fontweight='bold')
ax.set_xticks(range(len(fold_labels)))
ax.set_xticklabels(fold_labels, rotation=45, ha='right')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim([0, max(fold_values) * 1.1])

# Add value labels
for bar, val in zip(bars, fold_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()

# Save figure
plot_path = CHECKPOINT_DIR / 'ensemble_evaluation.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved visualization: {plot_path}")
plt.show()

print("\nPlots generated successfully.")

In [ ]:
# ==============================================================================
# Cell 9: Save Results for Thesis
# ==============================================================================

print("\n" + "="*70)
print(" SAVING RESULTS FOR THESIS")
print("="*70)

# 1. Save all predictions
predictions_df = pd.DataFrame({
    'id': all_ids,
    'fold': all_folds,
    'dataset': all_datasets,
    'true_label': all_labels,
    'predicted_probability': all_probs,
    'predicted_class': (all_probs >= primary_results['threshold']).astype(int)
})
predictions_path = CHECKPOINT_DIR / 'ensemble_predictions.csv'
predictions_df.to_csv(predictions_path, index=False)
print(f"\n1. Saved: {predictions_path.name} ({len(predictions_df):,} predictions)")

# 2. Save comprehensive metrics
metrics_summary = pd.DataFrame([{
    'tpr_5pct': metrics_dict['tpr_5pct'],
    'auroc': metrics_dict['auroc'],
    'auprc': metrics_dict['auprc'],
    'optimal_threshold': metrics_dict['threshold'],
    'accuracy': metrics_dict['accuracy'],
    'precision': metrics_dict['precision'],
    'recall': metrics_dict['recall'],
    'specificity': metrics_dict['specificity'],
    'f1_score': metrics_dict['f1_score'],
    'TP': metrics_dict['confusion_matrix_tp'],
    'TN': metrics_dict['confusion_matrix_tn'],
    'FP': metrics_dict['confusion_matrix_fp'],
    'FN': metrics_dict['confusion_matrix_fn'],
    'total_samples': metrics_dict['n_total'],
    'positive_samples': metrics_dict['n_positive'],
    'negative_samples': metrics_dict['n_negative'],
    'using_official_metrics': metrics_dict['using_official']
}])
summary_path = CHECKPOINT_DIR / 'ensemble_summary.csv'
metrics_summary.to_csv(summary_path, index=False)
print(f"2. Saved: {summary_path.name} (all metrics for thesis tables)")

# 3. Save threshold comparison
threshold_comparison = pd.DataFrame(results_by_threshold).T
threshold_path = CHECKPOINT_DIR / 'threshold_comparison.csv'
threshold_comparison.to_csv(threshold_path)
print(f"3. Saved: {threshold_path.name} (threshold analysis)")

# 4. Save per-dataset metrics
if dataset_metrics:
    dataset_df = pd.DataFrame(dataset_metrics).T
    dataset_path = CHECKPOINT_DIR / 'per_dataset_metrics.csv'
    dataset_df.to_csv(dataset_path)
    print(f"4. Saved: {dataset_path.name} (per-dataset performance)")

# 5. Save formatted text summary
summary_text = f"""CHAGASIGHT ENSEMBLE EVALUATION RESULTS
{'='*70}

Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

DATASET:
  Total samples: {metrics_dict['n_total']:,}
  Positive: {metrics_dict['n_positive']:,} ({100*metrics_dict['n_positive']/metrics_dict['n_total']:.2f}%)
  Negative: {metrics_dict['n_negative']:,} ({100*metrics_dict['n_negative']/metrics_dict['n_total']:.2f}%)

PRIMARY METRICS:
  TPR@5%:  {metrics_dict['tpr_5pct']:.4f}  (PRIMARY - Official PhysioNet)
  AUROC:   {metrics_dict['auroc']:.4f}
  AUPRC:   {metrics_dict['auprc']:.4f}

THRESHOLD-BASED METRICS (Threshold = {metrics_dict['threshold']:.4f}):
  Accuracy:    {metrics_dict['accuracy']:.4f}
  Precision:   {metrics_dict['precision']:.4f}
  Recall:      {metrics_dict['recall']:.4f}
  Specificity: {metrics_dict['specificity']:.4f}
  F1 Score:    {metrics_dict['f1_score']:.4f}

CONFUSION MATRIX:
                   Predicted
              Negative  Positive
  Actual    ┌─────────────────────┐
  Negative  │  {metrics_dict['confusion_matrix_tn']:7,}   {metrics_dict['confusion_matrix_fp']:7,}  │
  Positive  │  {metrics_dict['confusion_matrix_fn']:7,}   {metrics_dict['confusion_matrix_tp']:7,}  │
            └─────────────────────┘

PERFORMANCE BENCHMARKS:
  Random Baseline:      {RANDOM_BASELINE:.4f}
  Target Score:         {TARGET_SCORE:.4f}
  Top Team:             {TOP_TEAM:.4f}
  SOTA (Van Santvliet): {SOTA:.4f}
  
  Your Score:           {metrics_dict['tpr_5pct']:.4f}
  vs Target:            {'+' if metrics_dict['tpr_5pct'] >= TARGET_SCORE else ''}{metrics_dict['tpr_5pct'] - TARGET_SCORE:.4f}
  vs Top Team:          {'+' if metrics_dict['tpr_5pct'] >= TOP_TEAM else ''}{metrics_dict['tpr_5pct'] - TOP_TEAM:.4f}
  % of SOTA:            {100*metrics_dict['tpr_5pct']/SOTA:.1f}%

MODEL CONFIGURATION:
  Architecture: Hybrid dual-pathway (2D-ViT + 1D-ViT FM)
  Parameters: {total_params:,}
  Pretraining: MAE (2D) + ST-MEM (1D)
  Features: AoL, Demographics, REPA Alignment, Soft Labels
  Ensemble: 5-fold cross-validation

PAPER REFERENCES:
  1. Kim et al. (2025): Contour embedding, REPA alignment
  2. Van Santvliet et al. (2025): Foundation model, AoL, demographics
"""

summary_text_path = CHECKPOINT_DIR / 'EVALUATION_SUMMARY.txt'
with open(summary_text_path, 'w') as f:
    f.write(summary_text)
print(f"5. Saved: {summary_text_path.name} (formatted summary)")

print("\n" + "="*70)
print("ALL RESULTS SAVED SUCCESSFULLY")
print("="*70)
print(f"\nFiles saved in: {CHECKPOINT_DIR}")
print("\nUse these files for thesis Chapter 8.3:")
print("  - ensemble_summary.csv → Copy values to tables")
print("  - ensemble_evaluation.png → Figure 8.1")
print("  - per_dataset_metrics.csv → Per-dataset table")
print("  - EVALUATION_SUMMARY.txt → Reference for writing")
print("="*70)

In [ ]:
# ==============================================================================
# Cell 10: Package Final Ensemble Model for Deployment
# ==============================================================================

print("\n" + "="*70)
print(" PACKAGING ENSEMBLE MODEL FOR DEPLOYMENT")
print("="*70)

ensemble_checkpoint = {
    'ensemble_metrics': metrics_dict,
    'individual_fold_scores': fold_scores,
    'fold_models': [],
    'model_config': {
        'img_size': (24, 2048),
        'patch_size_2d': (8, 64),
        'num_leads': 12,
        'seq_len_1d': 1000,
        'patch_size_1d': 50,
        'embed_dim': 768,
        'depth': 12,
        'num_heads': 12,
        'use_aol': True,
        'use_demographics': True
    },
    'threshold': primary_results['threshold'],
    'dataset_statistics': {
        'n_total': metrics_dict['n_total'],
        'n_positive': metrics_dict['n_positive'],
        'n_negative': metrics_dict['n_negative']
    }
}

# Add each fold's model state
print("\nPackaging model states...")
for fold in range(5):
    checkpoint = torch.load(fold_checkpoints[fold], map_location='cpu', weights_only=False)
    ensemble_checkpoint['fold_models'].append({
        'fold': fold,
        'model_state_dict': checkpoint['model_state_dict'],
        'val_score': checkpoint.get('val_score', 0.0),
        'epoch': checkpoint.get('epoch', 0),
        'phase': checkpoint.get('phase', 'unknown')
    })
    print(f"  Fold {fold}: packed")

# Save ensemble model
ensemble_path = CHECKPOINT_DIR / 'FINAL_ENSEMBLE_MODEL.pt'
torch.save(ensemble_checkpoint, ensemble_path)

file_size_mb = ensemble_path.stat().st_size / 1024 / 1024
print(f"\n✓ Saved: {ensemble_path.name}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"  Contains: All 5 fold models + metrics + configuration")
print("\nThis file is deployment-ready!")
print("="*70)

# Evaluation Complete!

## Files Generated

All results saved in `checkpoints/` directory:

1. **ensemble_predictions.csv** - All predictions and labels
2. **ensemble_summary.csv** - Comprehensive metrics (use for thesis tables)
3. **ensemble_evaluation.png** - Visualization plots (use as Figure 8.1)
4. **per_dataset_metrics.csv** - Performance by dataset
5. **threshold_comparison.csv** - Metrics at different thresholds
6. **EVALUATION_SUMMARY.txt** - Formatted text summary
7. **FINAL_ENSEMBLE_MODEL.pt** - Complete ensemble model (deployment-ready)
8. **evaluation_checkpoints/** - Per-fold results (for resuming if interrupted)

## For Thesis Chapter 8.3

Use these files to create your thesis tables and figures:

**Table 1 (Primary Metrics)**: Values from `ensemble_summary.csv`
- TPR@5%, AUROC, AUPRC

**Table 2 (Confusion Matrix)**: Values from `ensemble_summary.csv`
- TP, TN, FP, FN

**Table 3 (Classification Metrics)**: Values from `ensemble_summary.csv`
- Accuracy, Precision, Recall, Specificity, F1

**Figure 8.1**: Use `ensemble_evaluation.png`
- Contains 4 subplots: ROC, PR curve, calibration, per-fold comparison

**Table 4 (Per-Dataset)**: Values from `per_dataset_metrics.csv`

## Next Steps

1. Copy values from CSV files to thesis tables
2. Insert ensemble_evaluation.png as figures
3. Write 2-3 paragraphs interpreting results
4. Submit thesis!

**Congratulations on completing your evaluation!**